# Medicare Part D — Colorado 2024: Prescriber Outlier Screening and Cost-per-Claim Regression

Runs top-to-bottom from the raw CMS CSV. Produces:

- `outputs/outliers.csv`
- `outputs/regression_output.csv`
- `outputs/PY_RESULTS.md`
- `outputs/PY_FLAGS.md`

Scope: calculations only. Primary metrics are cost-per-claim
(`Tot_Drug_Cst / Tot_Clms`) and cost-per-30-day-fill
(`Tot_Drug_Cst / Tot_30day_Fills`). Cost-per-beneficiary is not computed:
`Tot_Benes` is suppressed for cells under 11 beneficiaries, so the
denominator is missing non-randomly.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

CSV_NAME = 'Medicare_Part_D_Prescribers_by_Provider_and_Drug_2024.csv'

# Resolve paths whether the notebook is run from the project root or from outputs/.
_here = Path.cwd()
for _cand in [_here, _here.parent, *_here.parents]:
    if (_cand / CSV_NAME).exists():
        PROJECT_ROOT = _cand
        break
else:
    raise FileNotFoundError(f'Could not locate {CSV_NAME} at or above {_here}')

SRC = PROJECT_ROOT / CSV_NAME
OUT = PROJECT_ROOT / 'outputs'
OUT.mkdir(exist_ok=True)

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

print('source :', SRC)
print('outputs:', OUT)
print('pandas :', pd.__version__, '| numpy:', np.__version__)

source : C:\Users\Caleb Edwards\OneDrive\Documents\DAB Capstone\Medicare_Part_D_Prescribers_by_Provider_and_Drug_2024.csv
outputs: C:\Users\Caleb Edwards\OneDrive\Documents\DAB Capstone\outputs
pandas : 3.0.3 | numpy: 2.5.1


In [2]:
# Raw load. The file carries a UTF-8 BOM, hence utf-8-sig.
# No cleaning, no type coercion, no reshaping is applied to the source data.
USECOLS = [
    'Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name',
    'Prscrbr_City', 'Prscrbr_State_Abrvtn', 'Prscrbr_Type', 'Prscrbr_Type_Src',
    'Brnd_Name', 'Gnrc_Name', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Drug_Cst',
]
raw = pd.read_csv(SRC, encoding='utf-8-sig', usecols=USECOLS)

TOTAL_COST_SOURCE = float(raw['Tot_Drug_Cst'].sum())
N_ROWS_SOURCE = len(raw)

print(f'rows           : {N_ROWS_SOURCE:,}')
print(f'unique NPIs    : {raw["Prscrbr_NPI"].nunique():,}')
print(f'specialties    : {raw["Prscrbr_Type"].nunique():,}')
print(f'cities         : {raw["Prscrbr_City"].nunique():,}')
print(f'generic names  : {raw["Gnrc_Name"].nunique():,}')
print(f'states present : {sorted(raw["Prscrbr_State_Abrvtn"].unique())}')
print(f'total drug cost: ${TOTAL_COST_SOURCE:,.2f}')

rows           : 390,473
unique NPIs    : 19,390
specialties    : 97
cities         : 226
generic names  : 1,177
states present : ['CO']
total drug cost: $2,737,455,388.61


In [3]:
# Completeness of the columns used here (analysis columns only).
missing = raw[USECOLS].isna().sum().rename('n_missing').to_frame()
missing['pct_missing'] = 100 * missing['n_missing'] / len(raw)
display(missing)

# Boundary values in the metric inputs, recorded not corrected.
N_ZERO_COST_ROWS = int((raw['Tot_Drug_Cst'] == 0).sum())
N_NEG_COST_ROWS = int((raw['Tot_Drug_Cst'] < 0).sum())
print('rows with Tot_Drug_Cst == 0 :', N_ZERO_COST_ROWS)
print('rows with Tot_Drug_Cst <  0 :', N_NEG_COST_ROWS)
print('min Tot_Clms                :', int(raw['Tot_Clms'].min()))
print('min Tot_30day_Fills         :', float(raw['Tot_30day_Fills'].min()))

,n_missing,pct_missing
Prscrbr_NPI,0,0.0000
Prscrbr_Last_Org_Name,0,0.0000
Prscrbr_First_Name,0,0.0000
Prscrbr_City,0,0.0000
Prscrbr_State_Abrvtn,0,0.0000
Prscrbr_Type,0,0.0000
Prscrbr_Type_Src,0,0.0000
Brnd_Name,0,0.0000
Gnrc_Name,0,0.0000
Tot_Clms,0,0.0000


rows with Tot_Drug_Cst == 0 : 596
rows with Tot_Drug_Cst <  0 : 0
min Tot_Clms                : 11
min Tot_30day_Fills         : 11.0


## 1. Prescriber-level outlier screening within specialty

Aggregate the prescriber-drug rows to one row per NPI, compute both primary
metrics at that level, restrict to specialties with at least 30 prescribers,
then rank each prescriber against peers in the same specialty.

In [4]:
# Prescriber-level aggregation. Sums, not means: cost-per-claim is recomputed
# from summed numerator and summed denominator so that each prescriber's metric
# is claim-weighted across their drugs rather than an average of row ratios.
# Prscrbr_Type / Prscrbr_City / Prscrbr_Type_Src are constant within NPI in this
# file (verified below), so 'first' is a label carry-forward, not a choice.
_const_check = raw.groupby('Prscrbr_NPI')[['Prscrbr_Type', 'Prscrbr_City']].nunique()
assert (_const_check <= 1).all().all(), 'NPI maps to more than one specialty or city'

pres = (
    raw.groupby('Prscrbr_NPI', as_index=False)
       .agg(
           Prscrbr_Last_Org_Name=('Prscrbr_Last_Org_Name', 'first'),
           Prscrbr_First_Name=('Prscrbr_First_Name', 'first'),
           Prscrbr_City=('Prscrbr_City', 'first'),
           Prscrbr_Type=('Prscrbr_Type', 'first'),
           Prscrbr_Type_Src=('Prscrbr_Type_Src', 'first'),
           n_drug_rows=('Gnrc_Name', 'size'),
           n_distinct_generics=('Gnrc_Name', 'nunique'),
           Tot_Clms=('Tot_Clms', 'sum'),
           Tot_30day_Fills=('Tot_30day_Fills', 'sum'),
           Tot_Drug_Cst=('Tot_Drug_Cst', 'sum'),
       )
)

pres['cost_per_claim'] = pres['Tot_Drug_Cst'] / pres['Tot_Clms']
pres['cost_per_30day_fill'] = pres['Tot_Drug_Cst'] / pres['Tot_30day_Fills']

# Quality check: aggregation must be lossless on the cost total.
agg_total = float(pres['Tot_Drug_Cst'].sum())
assert abs(agg_total - TOTAL_COST_SOURCE) < 0.01, (agg_total, TOTAL_COST_SOURCE)
assert int(pres['Tot_Clms'].sum()) == int(raw['Tot_Clms'].sum())
assert len(pres) == raw['Prscrbr_NPI'].nunique()

N_PRESCRIBERS_ALL = len(pres)
print(f'prescribers                 : {N_PRESCRIBERS_ALL:,}')
print(f'total cost after aggregation: ${agg_total:,.2f}')
print(f'total cost in source        : ${TOTAL_COST_SOURCE:,.2f}')
print(f'difference                  : ${agg_total - TOTAL_COST_SOURCE:,.6f}')

prescribers                 : 19,390
total cost after aggregation: $2,737,455,388.61
total cost in source        : $2,737,455,388.61
difference                  : $0.000000


In [5]:
# Specialty restriction: at least 30 prescribers in the specialty, counted at
# prescriber level (not row level).
MIN_PRESCRIBERS_PER_SPECIALTY = 30

spec_counts = pres['Prscrbr_Type'].value_counts()
eligible_specialties = sorted(spec_counts[spec_counts >= MIN_PRESCRIBERS_PER_SPECIALTY].index)

N_SPECIALTIES_ALL = int(spec_counts.size)
N_SPECIALTIES_ELIGIBLE = len(eligible_specialties)

panel = pres[pres['Prscrbr_Type'].isin(eligible_specialties)].copy()
panel['specialty_n_prescribers'] = panel['Prscrbr_Type'].map(spec_counts)

N_PRESCRIBERS_PANEL = len(panel)

assert N_SPECIALTIES_ELIGIBLE == 46, N_SPECIALTIES_ELIGIBLE
print(f'specialties in file          : {N_SPECIALTIES_ALL}')
print(f'specialties with >= {MIN_PRESCRIBERS_PER_SPECIALTY} prescribers: {N_SPECIALTIES_ELIGIBLE}')
print(f'prescribers retained         : {N_PRESCRIBERS_PANEL:,} of {N_PRESCRIBERS_ALL:,} '
      f'({100 * N_PRESCRIBERS_PANEL / N_PRESCRIBERS_ALL:.2f}%)')
print(f'cost covered by retained set : ${panel["Tot_Drug_Cst"].sum():,.2f} '
      f'({100 * panel["Tot_Drug_Cst"].sum() / TOTAL_COST_SOURCE:.2f}% of source total)')

specialties in file          : 97
specialties with >= 30 prescribers: 46
prescribers retained         : 18,940 of 19,390 (97.68%)
cost covered by retained set : $2,655,447,283.20 (97.00% of source total)


In [6]:
# OUTLIER RULE (stated explicitly, applied identically to both primary metrics)
# -----------------------------------------------------------------------------
# Rule: Tukey IQR fences computed WITHIN each specialty.
#   Q1, Q3 = 25th and 75th percentile of the metric among prescribers in that
#            specialty (linear interpolation, pandas default)
#   IQR    = Q3 - Q1
#   upper fence = Q3 + 1.5 * IQR      lower fence = Q1 - 1.5 * IQR
#   A prescriber is flagged HIGH if metric > upper fence, LOW if metric < lower
#   fence. High and low are flagged separately; both are reported.
#
# Why IQR and not z-score: both metrics are strongly right-skewed and heavy
# tailed at prescriber level (max cost-per-claim exceeds the median by more than
# three orders of magnitude). A z-score rule uses the mean and standard
# deviation, and both of those statistics are themselves inflated by the extreme
# values the rule is meant to detect, so a small number of very large values
# raises the threshold and hides the rest (masking). Quartiles do not move when
# a tail value moves further out, so the fences stay stable. This is a
# methodological judgment, recorded as such in PY_FLAGS.md.
#
# Note the two rules are not interchangeable here: 1.5*IQR is not a fixed number
# of standard deviations except under normality, which does not hold for these
# distributions. Percentile rank is reported alongside the flag so the position
# of every prescriber is visible regardless of the cutoff chosen.
IQR_MULTIPLIER = 1.5
METRICS = ['cost_per_claim', 'cost_per_30day_fill']

for m in METRICS:
    g = panel.groupby('Prscrbr_Type')[m]
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    panel[f'{m}_spec_q1'] = q1
    panel[f'{m}_spec_median'] = g.transform('median')
    panel[f'{m}_spec_q3'] = q3
    panel[f'{m}_spec_iqr'] = iqr
    panel[f'{m}_upper_fence'] = q3 + IQR_MULTIPLIER * iqr
    panel[f'{m}_lower_fence'] = q1 - IQR_MULTIPLIER * iqr
    # Percentile rank within specialty, 0-100, ties averaged.
    panel[f'{m}_pctile_in_specialty'] = 100 * g.rank(pct=True, method='average')
    panel[f'{m}_flag_high'] = panel[m] > panel[f'{m}_upper_fence']
    panel[f'{m}_flag_low'] = panel[m] < panel[f'{m}_lower_fence']

panel['flag_any_high'] = panel[[f'{m}_flag_high' for m in METRICS]].any(axis=1)
panel['flag_both_high'] = panel[[f'{m}_flag_high' for m in METRICS]].all(axis=1)
panel['flag_any_low'] = panel[[f'{m}_flag_low' for m in METRICS]].any(axis=1)
panel['n_flags'] = panel[[f'{m}_flag_high' for m in METRICS]
                         + [f'{m}_flag_low' for m in METRICS]].sum(axis=1)

flag_summary = pd.DataFrame({
    'metric': [m for m in METRICS for _ in (0, 1)] + ['any_metric'] * 3,
    'direction': ['high', 'low'] * len(METRICS) + ['high', 'low', 'high_on_both'],
    'n_flagged': [
        int(panel['cost_per_claim_flag_high'].sum()),
        int(panel['cost_per_claim_flag_low'].sum()),
        int(panel['cost_per_30day_fill_flag_high'].sum()),
        int(panel['cost_per_30day_fill_flag_low'].sum()),
        int(panel['flag_any_high'].sum()),
        int(panel['flag_any_low'].sum()),
        int(panel['flag_both_high'].sum()),
    ],
})
flag_summary['pct_of_panel'] = 100 * flag_summary['n_flagged'] / N_PRESCRIBERS_PANEL
display(flag_summary)

,metric,direction,n_flagged,pct_of_panel
0,cost_per_claim,high,1752,9.2503
1,cost_per_claim,low,13,0.0686
2,cost_per_30day_fill,high,1805,9.5301
3,cost_per_30day_fill,low,11,0.0581
4,any_metric,high,1976,10.4329
5,any_metric,low,13,0.0686
6,any_metric,high_on_both,1581,8.3474


In [7]:
# Distribution of both metrics at prescriber level, over the retained panel.
metric_dist = panel[METRICS].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
metric_dist['skew'] = [panel[m].skew() for m in METRICS]
metric_dist['kurtosis'] = [panel[m].kurtosis() for m in METRICS]
display(metric_dist)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,skew,kurtosis
cost_per_claim,"18,940.0000",179.2756,924.3280,0.0000,1.7975,3.0083,7.4085,31.6903,97.8655,644.1490,"2,868.2445","70,720.0000",35.1877,"2,158.4568"
cost_per_30day_fill,"18,940.0000",124.7808,829.8845,0.0000,1.6703,2.7407,6.0364,17.9179,55.7383,416.4078,"1,997.0941","70,720.0000",45.3956,"3,243.1324"


In [8]:
# Per-specialty rule parameters and flag counts. These are the fences the rule
# above uses; they are distribution statistics of the metric, not cost totals.
spec_rule = (
    panel.groupby('Prscrbr_Type')
         .agg(
             n_prescribers=('Prscrbr_NPI', 'size'),
             cpc_q1=('cost_per_claim_spec_q1', 'first'),
             cpc_median=('cost_per_claim_spec_median', 'first'),
             cpc_q3=('cost_per_claim_spec_q3', 'first'),
             cpc_upper_fence=('cost_per_claim_upper_fence', 'first'),
             cpc_n_high=('cost_per_claim_flag_high', 'sum'),
             cp30_q1=('cost_per_30day_fill_spec_q1', 'first'),
             cp30_median=('cost_per_30day_fill_spec_median', 'first'),
             cp30_q3=('cost_per_30day_fill_spec_q3', 'first'),
             cp30_upper_fence=('cost_per_30day_fill_upper_fence', 'first'),
             cp30_n_high=('cost_per_30day_fill_flag_high', 'sum'),
         )
         .reset_index()
         .sort_values('n_prescribers', ascending=False)
)
spec_rule['cpc_pct_high'] = 100 * spec_rule['cpc_n_high'] / spec_rule['n_prescribers']
spec_rule['cp30_pct_high'] = 100 * spec_rule['cp30_n_high'] / spec_rule['n_prescribers']
print('specialties in table:', len(spec_rule))
display(spec_rule.head(15))

specialties in table: 46


,Prscrbr_Type,n_prescribers,cpc_q1,cpc_median,cpc_q3,cpc_upper_fence,cpc_n_high,cp30_q1,cp30_median,cp30_q3,cp30_upper_fence,cp30_n_high,cpc_pct_high,cp30_pct_high
24,Nurse Practitioner,3509,12.3216,49.0310,110.1624,256.9235,391,9.3007,29.2472,72.2905,166.7753,378,11.1428,10.7723
34,Physician Assistant,3010,7.3786,18.6042,80.2114,189.4607,366,6.4258,13.7711,45.1325,103.1925,398,12.1595,13.2226
10,Family Practice,2288,28.2859,55.1870,82.9226,164.8777,99,13.3213,24.7572,39.7857,79.4822,130,4.3269,5.6818
6,Dentist,1986,2.9922,3.8870,4.9837,7.9710,95,2.9840,3.8683,4.9554,7.9126,92,4.7835,4.6324
18,Internal Medicine,1292,28.8849,65.0009,106.6994,223.4211,137,14.5510,31.2224,58.3049,123.9357,147,10.6037,11.3777
8,Emergency Medicine,708,4.8881,6.5057,9.0036,15.1770,79,4.8596,6.4446,8.8363,14.8014,74,11.1582,10.4520
27,Optometry,523,12.4032,32.5852,144.8799,343.5949,56,6.8872,19.3863,85.9631,204.5769,63,10.7075,12.0459
25,Obstetrics & Gynecology,410,40.0844,62.8108,100.0300,189.9484,29,16.3582,27.1023,44.0651,85.6255,27,7.0732,6.5854
29,Orthopedic Surgery,367,4.7667,6.6939,8.9888,15.3219,37,4.2475,6.2937,8.4633,14.7870,31,10.0817,8.4469
26,Ophthalmology,315,25.0434,45.9009,94.3095,198.2086,24,16.7755,30.4650,56.5729,116.2688,27,7.6190,8.5714


In [9]:
# Assemble and export the outlier table: every prescriber in the retained panel,
# with both metrics, both percentile ranks, the fences applied, and the flags.
OUTLIER_COLS = [
    'Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City',
    'Prscrbr_Type', 'Prscrbr_Type_Src', 'specialty_n_prescribers',
    'n_drug_rows', 'n_distinct_generics',
    'Tot_Clms', 'Tot_30day_Fills', 'Tot_Drug_Cst',
    'cost_per_claim', 'cost_per_claim_pctile_in_specialty',
    'cost_per_claim_spec_q1', 'cost_per_claim_spec_median', 'cost_per_claim_spec_q3',
    'cost_per_claim_spec_iqr', 'cost_per_claim_lower_fence', 'cost_per_claim_upper_fence',
    'cost_per_claim_flag_high', 'cost_per_claim_flag_low',
    'cost_per_30day_fill', 'cost_per_30day_fill_pctile_in_specialty',
    'cost_per_30day_fill_spec_q1', 'cost_per_30day_fill_spec_median',
    'cost_per_30day_fill_spec_q3', 'cost_per_30day_fill_spec_iqr',
    'cost_per_30day_fill_lower_fence', 'cost_per_30day_fill_upper_fence',
    'cost_per_30day_fill_flag_high', 'cost_per_30day_fill_flag_low',
    'flag_any_high', 'flag_both_high', 'flag_any_low', 'n_flags',
]
outliers = (panel[OUTLIER_COLS]
            .sort_values(['Prscrbr_Type', 'cost_per_claim_pctile_in_specialty'],
                         ascending=[True, False])
            .reset_index(drop=True))
outliers.to_csv(OUT / 'outliers.csv', index=False)
print(f'wrote {OUT / "outliers.csv"}  rows={len(outliers):,} cols={outliers.shape[1]}')

top_cpc = (panel.sort_values('cost_per_claim', ascending=False)
                .head(20)[['Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name',
                           'Prscrbr_City', 'Prscrbr_Type', 'Tot_Clms', 'Tot_Drug_Cst',
                           'cost_per_claim', 'cost_per_claim_pctile_in_specialty',
                           'cost_per_claim_upper_fence', 'cost_per_claim_flag_high']]
                .reset_index(drop=True))
top_cp30 = (panel.sort_values('cost_per_30day_fill', ascending=False)
                 .head(20)[['Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name',
                            'Prscrbr_City', 'Prscrbr_Type', 'Tot_30day_Fills', 'Tot_Drug_Cst',
                            'cost_per_30day_fill', 'cost_per_30day_fill_pctile_in_specialty',
                            'cost_per_30day_fill_upper_fence', 'cost_per_30day_fill_flag_high']]
                 .reset_index(drop=True))
display(top_cpc)
display(top_cp30)

wrote C:\Users\Caleb Edwards\OneDrive\Documents\DAB Capstone\outputs\outliers.csv  rows=18,940 cols=36


,Prscrbr_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_City,Prscrbr_Type,Tot_Clms,Tot_Drug_Cst,cost_per_claim,cost_per_claim_pctile_in_specialty,cost_per_claim_upper_fence,cost_per_claim_flag_high
0,1083707434,Apkon,Susan,Aurora,Physical Medicine and Rehabilitation,16,"1,131,520.0000","70,720.0000",100.0000,93.6114,True
1,1013007780,Kirkpatrick,Charles,Aurora,Allergy/ Immunology,13,"563,929.2300","43,379.1715",100.0000,"1,404.4410",True
2,1245728203,King,Mary,Centennial,Physician Assistant,196,"6,045,733.9400","30,845.5813",100.0000,189.4607,True
3,1407374408,Swint,Carly,Centennial,Physician Assistant,316,"5,965,211.8400","18,877.2527",99.9668,189.4607,True
4,1366746893,Hoffman,Lindsey,Aurora,Pediatric Medicine,26,"430,127.4900","16,543.3650",100.0000,628.0043,True
5,1871286435,Johnson,Kelli,Aurora,Nurse Practitioner,30,"495,079.4500","16,502.6483",100.0000,256.9235,True
6,1255532990,Yeh,Daniel,Denver,General Surgery,39,"588,211.9700","15,082.3582",100.0000,18.2860,True
7,1235259169,Gross,Jane,Denver,Pediatric Medicine,65,"938,693.6000","14,441.4400",97.2973,628.0043,True
8,1922797612,Reinhart,Anna,Aurora,Nurse Practitioner,81,"1,124,047.5900","13,877.1307",99.9715,256.9235,True
9,1164842241,Plost,Grant,Aurora,Dermatology,11,"148,365.8600","13,487.8055",100.0000,714.4214,True


,Prscrbr_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_City,Prscrbr_Type,Tot_30day_Fills,Tot_Drug_Cst,cost_per_30day_fill,cost_per_30day_fill_pctile_in_specialty,cost_per_30day_fill_upper_fence,cost_per_30day_fill_flag_high
0,1083707434,Apkon,Susan,Aurora,Physical Medicine and Rehabilitation,16.0000,"1,131,520.0000","70,720.0000",100.0000,86.6731,True
1,1013007780,Kirkpatrick,Charles,Aurora,Allergy/ Immunology,13.0000,"563,929.2300","43,379.1715",100.0000,959.2867,True
2,1245728203,King,Mary,Centennial,Physician Assistant,264.1000,"6,045,733.9400","22,891.8362",100.0000,103.1925,True
3,1407374408,Swint,Carly,Centennial,Physician Assistant,350.5000,"5,965,211.8400","17,019.1493",99.9668,103.1925,True
4,1366746893,Hoffman,Lindsey,Aurora,Pediatric Medicine,26.0000,"430,127.4900","16,543.3650",100.0000,574.7394,True
5,1255532990,Yeh,Daniel,Denver,General Surgery,39.0000,"588,211.9700","15,082.3582",100.0000,15.1435,True
6,1871286435,Johnson,Kelli,Aurora,Nurse Practitioner,34.0000,"495,079.4500","14,561.1603",100.0000,166.7753,True
7,1235259169,Gross,Jane,Denver,Pediatric Medicine,65.6000,"938,693.6000","14,309.3537",97.2973,574.7394,True
8,1922797612,Reinhart,Anna,Aurora,Nurse Practitioner,83.0000,"1,124,047.5900","13,542.7420",99.9715,166.7753,True
9,1164842241,Plost,Grant,Aurora,Dermatology,11.0000,"148,365.8600","13,487.8055",100.0000,600.7917,True


## 2. OLS regression of cost-per-claim on specialty, city, and generic drug

Specification and assumptions are stated in the cell comments below.

In [10]:
# SPECIFICATION
# -----------------------------------------------------------------------------
# Unit of observation: the prescriber-drug row as it appears in the source file
#   (n = 390,473). Generic drug is a row-level attribute, so it cannot enter a
#   prescriber-level regression; the outlier section above is prescriber-level
#   and this section is row-level. The two units are different by necessity.
#
# Model:  cpc_i = b0 + a[specialty(i)] + c[city(i)] + d[generic(i)] + e_i
#   cpc_i = Tot_Drug_Cst_i / Tot_Clms_i, in dollars, untransformed.
#   All three predictors are categorical, entered as full sets of indicator
#   variables with one reference level each (first level alphabetically).
#   Additive main effects only; no interactions; no continuous controls.
#   Estimated by ordinary least squares, unweighted (each row counts once,
#   whether it carries 11 claims or 50,000).
#   Levels: 97 specialties + 226 cities + 1,177 generics = 1,500 category levels;
#   with one reference level omitted per dimension the model carries 1,498 free
#   parameters including the intercept.
#
# ASSUMPTIONS THIS SPECIFICATION MAKES (none are tested here)
#   1. Linearity/additivity: a specialty shift is the same dollar amount in every
#      city and for every drug.
#   2. Independence of errors across rows. This is violated by construction --
#      a prescriber contributes many rows (median ~14) and a drug appears across
#      many prescribers. No clustering or panel correction is applied.
#   3. Homoskedasticity. Almost certainly violated given the outcome's skew.
#   4. Errors mean-zero conditional on the regressors, which is what OLS needs
#      for the coefficients to be conditional means. Nothing here identifies a
#      causal effect: this is a descriptive variance decomposition, and the
#      coefficients are conditional mean differences, not effects of anything.
#   5. The outcome is treated as continuous and unbounded; in fact it is bounded
#      below by 0 and 596 rows sit exactly at that bound.
#
# ESTIMATION METHOD
#   A dense design matrix would be 390,473 x 1,500 (~4.7 GB), and scipy sparse
#   is not available in this environment, so the model is fit by alternating
#   projections (block coordinate descent / backfitting over the three factors).
#   Each block update solves exactly for one factor's effects given the others,
#   which is a groupwise mean of the partial residual. The algorithm converges to
#   the OLS solution for additive categorical models; convergence is verified
#   below by checking the normal equations directly -- at the OLS optimum the
#   residuals must sum to zero within every level of every factor.
reg = pd.DataFrame({
    'y': (raw['Tot_Drug_Cst'] / raw['Tot_Clms']).astype(float),
})
FACTOR_COLS = {'specialty': 'Prscrbr_Type', 'city': 'Prscrbr_City', 'generic': 'Gnrc_Name'}

codes, levels = {}, {}
for fac, col in FACTOR_COLS.items():
    c, l = pd.factorize(raw[col], sort=True)
    codes[fac] = np.asarray(c, dtype=np.intp)
    levels[fac] = np.asarray(l, dtype=object)
    print(f'{fac:10s} levels={len(l):5d}  reference={l[0]!r}')

y = reg['y'].to_numpy(float)
N_OBS = y.size
TSS = float(((y - y.mean()) ** 2).sum())
print(f'\nn = {N_OBS:,}   mean(y) = {y.mean():,.4f}   sd(y) = {y.std(ddof=1):,.4f}   TSS = {TSS:,.2f}')

specialty  levels=   97  reference='Addiction Medicine'
city       levels=  226  reference='Akron'
generic    levels= 1177  reference='0.9 % Sodium Chloride'

n = 390,473   mean(y) = 230.6018   sd(y) = 1,427.5396   TSS = 795,730,927,547.66


In [11]:
def fit_additive_ols(factor_names, y, codes, tol=1e-10, max_iter=5000):
    """OLS of y on an additive set of categorical factors, by alternating projections.

    Returns (intercept, effects dict, fitted values, rss, iterations, max normal-
    equation violation). Each factor's effects are centered to mean zero across
    its levels, with the intercept absorbing the shift; the reference-level
    recoding happens afterwards.
    """
    factor_names = list(factor_names)
    counts = {f: np.bincount(codes[f]).astype(float) for f in factor_names}
    eff = {f: np.zeros(counts[f].size) for f in factor_names}
    mu = float(y.mean())
    fitted = np.full(y.size, mu)
    iters, delta = 0, np.inf
    for it in range(max_iter):
        prev = fitted.copy()
        for f in factor_names:
            partial = y - mu
            for g in factor_names:
                if g != f:
                    partial -= eff[g][codes[g]]
            e = np.bincount(codes[f], weights=partial) / counts[f]
            m = float(e.mean())
            eff[f] = e - m
            mu += m
        fitted = np.full(y.size, mu)
        for f in factor_names:
            fitted += eff[f][codes[f]]
        delta = float(np.abs(fitted - prev).max())
        iters = it + 1
        if delta < tol:
            break
    resid = y - fitted
    rss = float((resid ** 2).sum())
    # Normal-equation check: at the optimum, sum of residuals within each level
    # of each factor is zero. Reported relative to the scale of |y|.
    ne = max(float(np.abs(np.bincount(codes[f], weights=resid)).max()) for f in factor_names)
    return mu, eff, fitted, rss, iters, delta, ne


def model_stats(name, factor_names, rss, n_params):
    r2 = 1 - rss / TSS
    adj = 1 - (1 - r2) * (N_OBS - 1) / (N_OBS - n_params)
    return {
        'model': name,
        'n_factors': len(factor_names),
        'n_params': n_params,
        'rss': rss,
        'r_squared': r2,
        'adj_r_squared': adj,
        'rmse': np.sqrt(rss / (N_OBS - n_params)),
    }


N_LEVELS = {f: len(levels[f]) for f in FACTOR_COLS}
MODELS = [
    ('specialty',), ('city',), ('generic',),
    ('specialty', 'city'), ('specialty', 'generic'), ('city', 'generic'),
    ('specialty', 'city', 'generic'),
]

fits, rows = {}, []
for combo in MODELS:
    mu, eff, fitted, rss, iters, delta, ne = fit_additive_ols(combo, y, codes)
    n_params = 1 + sum(N_LEVELS[f] - 1 for f in combo)
    fits[combo] = dict(mu=mu, eff=eff, fitted=fitted, rss=rss)
    st = model_stats(' + '.join(combo), combo, rss, n_params)
    st.update(iterations=iters, max_fitted_change=delta, normal_eq_max_abs=ne)
    rows.append(st)

model_table = pd.DataFrame(rows)
display(model_table)

FULL = ('specialty', 'city', 'generic')
R2_FULL = float(model_table.loc[model_table['model'] == ' + '.join(FULL), 'r_squared'].iloc[0])
print(f'\nfull-model R-squared: {R2_FULL:.6f}')
print('max |sum of residuals within a level| across models: '
      f'{model_table["normal_eq_max_abs"].max():.3e}  (TSS scale: {TSS:,.0f})')

,model,n_factors,n_params,rss,r_squared,adj_r_squared,rmse,iterations,max_fitted_change,normal_eq_max_abs
0,specialty,1,97,"727,415,761,496.9479",0.0859,0.0856,"1,365.0539",2,0.0000,0.0000
1,city,1,226,"793,150,522,248.7919",0.0032,0.0027,"1,425.6339",2,0.0000,0.0000
2,generic,1,1177,"56,197,242,454.5465",0.9294,0.9292,379.9422,2,0.0000,0.0000
3,specialty + city,2,322,"726,912,782,714.2671",0.0865,0.0857,"1,364.9753",29,0.0000,0.0000
4,specialty + generic,2,1273,"56,048,772,325.3095",0.9296,0.9293,379.4868,101,0.0000,0.0000
5,city + generic,2,1402,"56,142,627,774.6851",0.9294,0.9292,379.8673,11,0.0000,0.0000
6,specialty + city + generic,3,1498,"55,999,249,251.2203",0.9296,0.9294,379.4288,101,0.0000,0.0000



full-model R-squared: 0.929625
max |sum of residuals within a level| across models: 7.154e-07  (TSS scale: 795,730,927,548)


In [12]:
# VARIANCE-EXPLAINED COMPARISON ACROSS THE THREE DIMENSIONS
# Two complementary measures, because the dimensions are correlated with one
# another and so single-factor R-squareds do not add up to the full-model value:
#   own_r_squared         - R-squared of that dimension alone (its total share,
#                           including whatever it shares with the others)
#   incremental_r_squared - full-model R-squared minus the R-squared of the model
#                           with the other two dimensions only (its unique share,
#                           net of overlap)
# adj_own_r_squared is reported because a factor with more levels mechanically
# fits better: 1,177 generic levels versus 97 specialties and 226 cities.
def r2_of(combo):
    return 1 - fits[tuple(combo)]['rss'] / TSS


var_rows = []
for f in FULL:
    others = tuple(x for x in FULL if x != f)
    own = r2_of((f,))
    n_params_own = N_LEVELS[f]
    var_rows.append({
        'dimension': f,
        'source_column': FACTOR_COLS[f],
        'n_levels': N_LEVELS[f],
        'own_r_squared': own,
        'adj_own_r_squared': 1 - (1 - own) * (N_OBS - 1) / (N_OBS - n_params_own),
        'r_squared_other_two': r2_of(others),
        'incremental_r_squared': R2_FULL - r2_of(others),
        'pct_of_full_model_r2_own': 100 * own / R2_FULL,
    })
variance_table = pd.DataFrame(var_rows).sort_values('own_r_squared', ascending=False)
display(variance_table)

shared = float(variance_table['own_r_squared'].sum() - R2_FULL)
print(f'sum of single-dimension R-squareds : {variance_table["own_r_squared"].sum():.6f}')
print(f'full-model R-squared               : {R2_FULL:.6f}')
print(f'sum minus full (overlap indicator) : {shared:.6f}')
print(f'unexplained share of variance      : {1 - R2_FULL:.6f}')

win_own = variance_table.loc[variance_table['own_r_squared'].idxmax()]
win_inc = variance_table.loc[variance_table['incremental_r_squared'].idxmax()]
print(f'\nMost variance explained, single-dimension R-squared: '
      f'{win_own["dimension"]} (R2 = {win_own["own_r_squared"]:.6f})')
print(f'Most variance explained, incremental R-squared     : '
      f'{win_inc["dimension"]} (incremental R2 = {win_inc["incremental_r_squared"]:.6f})')
print(f'Both measures agree: {win_own["dimension"] == win_inc["dimension"]}')
MOST_VARIANCE_DIM = win_own['dimension']

,dimension,source_column,n_levels,own_r_squared,adj_own_r_squared,r_squared_other_two,incremental_r_squared,pct_of_full_model_r2_own
2,generic,Gnrc_Name,1177,0.9294,0.9292,0.0865,0.8431,99.9732
0,specialty,Prscrbr_Type,97,0.0859,0.0856,0.9294,0.0002,9.2351
1,city,Prscrbr_City,226,0.0032,0.0027,0.9296,0.0001,0.3488


sum of single-dimension R-squareds : 1.018471
full-model R-squared               : 0.929625
sum minus full (overlap indicator) : 0.088846
unexplained share of variance      : 0.070375

Most variance explained, single-dimension R-squared: generic (R2 = 0.929377)
Most variance explained, incremental R-squared     : generic (incremental R2 = 0.843141)
Both measures agree: True


In [13]:
# Coefficients from the full three-factor model, converted from mean-centered
# effects to reference-level (dummy) coding: the first level of each factor,
# alphabetically, is the omitted reference and its coefficient is folded into the
# intercept. A coefficient is therefore the conditional mean difference in
# dollars-per-claim versus that reference level, holding the other two
# dimensions fixed.
# No standard errors, t-statistics, or p-values are produced: those require
# (X'X)^-1 for a 1,498-parameter design, which alternating projections does not
# compute and which is not feasible here without a sparse solver.
full = fits[FULL]
intercept = full['mu']
coef_rows = []
for f in FULL:
    e = full['eff'][f]
    ref_val = e[0]
    intercept += ref_val
    counts = np.bincount(codes[f], minlength=len(levels[f]))
    for i, lv in enumerate(levels[f]):
        coef_rows.append({
            'block': f,
            'source_column': FACTOR_COLS[f],
            'level': str(lv),
            'is_reference_level': i == 0,
            'coefficient': float(e[i] - ref_val),
            'n_obs_at_level': int(counts[i]),
        })

coefs = pd.DataFrame(coef_rows)
coefs = pd.concat([
    pd.DataFrame([{'block': 'intercept', 'source_column': '', 'level': '(Intercept)',
                   'is_reference_level': False, 'coefficient': float(intercept),
                   'n_obs_at_level': N_OBS}]),
    coefs,
], ignore_index=True)

# Verify the recoded coefficients reproduce the fitted values exactly.
_check = np.full(N_OBS, intercept)
for f in FULL:
    e = full['eff'][f]
    _check += (e - e[0])[codes[f]]
assert np.abs(_check - full['fitted']).max() < 1e-6, np.abs(_check - full['fitted']).max()
print('recoded coefficients reproduce fitted values, max abs diff:',
      f'{np.abs(_check - full["fitted"]).max():.2e}')
print(f'coefficient rows: {len(coefs):,}')
display(coefs.groupby('block').agg(n=('coefficient', 'size'),
                                   min=('coefficient', 'min'),
                                   median=('coefficient', 'median'),
                                   max=('coefficient', 'max')))

recoded coefficients reproduce fitted values, max abs diff: 1.46e-11
coefficient rows: 1,501


,n,min,median,max
block,,,,
city,226,-418.2847,44.7516,324.4193
generic,1177,-95.3373,143.7117,"121,860.8778"
intercept,1,-59.1553,-59.1553,-59.1553
specialty,97,-593.1952,19.0623,506.8974


In [14]:
# regression_output.csv carries both the coefficient table and the model-level
# statistics, distinguished by the `block` column, so that the whole regression
# output travels in the single named file.
stat_rows = []
for _, r in model_table.iterrows():
    for k in ['r_squared', 'adj_r_squared', 'rss', 'rmse', 'n_params',
              'iterations', 'normal_eq_max_abs']:
        stat_rows.append({'block': 'model_fit', 'source_column': r['model'],
                          'level': k, 'is_reference_level': False,
                          'coefficient': float(r[k]), 'n_obs_at_level': N_OBS})
for _, r in variance_table.iterrows():
    for k in ['n_levels', 'own_r_squared', 'adj_own_r_squared',
              'r_squared_other_two', 'incremental_r_squared', 'pct_of_full_model_r2_own']:
        stat_rows.append({'block': 'variance_explained', 'source_column': r['dimension'],
                          'level': k, 'is_reference_level': False,
                          'coefficient': float(r[k]), 'n_obs_at_level': N_OBS})
stat_rows.append({'block': 'model_fit', 'source_column': 'specialty + city + generic',
                  'level': 'n_obs', 'is_reference_level': False,
                  'coefficient': float(N_OBS), 'n_obs_at_level': N_OBS})
stat_rows.append({'block': 'model_fit', 'source_column': 'specialty + city + generic',
                  'level': 'total_sum_of_squares', 'is_reference_level': False,
                  'coefficient': TSS, 'n_obs_at_level': N_OBS})

regression_output = pd.concat([pd.DataFrame(stat_rows), coefs], ignore_index=True)
regression_output.to_csv(OUT / 'regression_output.csv', index=False)
print(f'wrote {OUT / "regression_output.csv"}  rows={len(regression_output):,} '
      f'cols={regression_output.shape[1]}')
display(regression_output['block'].value_counts().rename('rows').to_frame())

wrote C:\Users\Caleb Edwards\OneDrive\Documents\DAB Capstone\outputs\regression_output.csv  rows=1,570 cols=6


,rows
block,
generic,1177
city,226
specialty,97
model_fit,51
variance_explained,18
intercept,1


## 3. Markdown exports

`PY_RESULTS.md` holds the numbers as plain tables. `PY_FLAGS.md` holds data
quality issues, foreseeable transformations, specification concerns,
ambiguities, and interpretation labeled as such.

In [15]:
ID_COLS = ('Prscrbr_NPI',)  # identifiers: printed as digits, never comma-grouped


def to_md(df, floatfmt='{:,.4f}', max_rows=None):
    """Render a DataFrame as a GitHub-flavoured markdown table, no dependencies."""
    d = df if max_rows is None else df.head(max_rows)
    def cell(v, col):
        if col in ID_COLS:
            return str(v)
        if isinstance(v, (bool, np.bool_)):
            return str(bool(v))
        if isinstance(v, (int, np.integer)):
            return f'{int(v):,}'
        if isinstance(v, (float, np.floating)):
            if not np.isfinite(v):
                return str(v)
            if v != 0 and abs(v) < 1e-4:
                return f'{v:.3e}'   # keep convergence diagnostics readable
            return floatfmt.format(v)
        return str(v)
    cols = list(d.columns)
    head = '| ' + ' | '.join(str(c) for c in cols) + ' |'
    rule = '| ' + ' | '.join('---' for _ in cols) + ' |'
    body = ['| ' + ' | '.join(cell(v, c) for v, c in zip(row, cols)) + ' |'
            for row in d.itertuples(index=False, name=None)]
    return '\n'.join([head, rule] + body)


print(to_md(variance_table))

| dimension | source_column | n_levels | own_r_squared | adj_own_r_squared | r_squared_other_two | incremental_r_squared | pct_of_full_model_r2_own |
| --- | --- | --- | --- | --- | --- | --- | --- |
| generic | Gnrc_Name | 1,177 | 0.9294 | 0.9292 | 0.0865 | 0.8431 | 99.9732 |
| specialty | Prscrbr_Type | 97 | 0.0859 | 0.0856 | 0.9294 | 0.0002 | 9.2351 |
| city | Prscrbr_City | 226 | 0.0032 | 0.0027 | 0.9296 | 6.224e-05 | 0.3488 |


In [16]:
# Pre-rendered table blocks and scalars, kept out of the f-string below.
metric_dist_out = metric_dist.reset_index(names='metric')
coefs_specialty_out = (coefs[coefs['block'].isin(['intercept', 'specialty'])]
                       .sort_values(['block', 'coefficient'], ascending=[True, False]))
city_extremes_out = pd.concat([coefs[coefs['block'] == 'city'].nlargest(20, 'coefficient'),
                               coefs[coefs['block'] == 'city'].nsmallest(20, 'coefficient')])
generic_extremes_out = pd.concat([coefs[coefs['block'] == 'generic'].nlargest(25, 'coefficient'),
                                  coefs[coefs['block'] == 'generic'].nsmallest(25, 'coefficient')])
REF_SPECIALTY = str(levels['specialty'][0])
REF_CITY = str(levels['city'][0])
REF_GENERIC = str(levels['generic'][0])
PANEL_COST_PCT = 100 * panel['Tot_Drug_Cst'].sum() / TOTAL_COST_SOURCE
WIN_OWN_DIM, WIN_OWN_R2 = win_own['dimension'], win_own['own_r_squared']
WIN_INC_DIM, WIN_INC_R2 = win_inc['dimension'], win_inc['incremental_r_squared']
FULL_N_PARAMS = int(model_table.loc[model_table['model'] == ' + '.join(FULL), 'n_params'].iloc[0])
FULL_N_LEVELS = sum(N_LEVELS.values())

run_facts = pd.DataFrame([
    ('source_rows', f'{N_ROWS_SOURCE:,}'),
    ('source_total_drug_cost_usd', f'{TOTAL_COST_SOURCE:,.2f}'),
    ('prescribers_all', f'{N_PRESCRIBERS_ALL:,}'),
    ('prescriber_level_total_drug_cost_usd', f'{agg_total:,.2f}'),
    ('aggregation_cost_difference_usd', f'{agg_total - TOTAL_COST_SOURCE:,.6f}'),
    ('specialties_in_file', f'{N_SPECIALTIES_ALL}'),
    ('specialty_min_prescribers_threshold', f'{MIN_PRESCRIBERS_PER_SPECIALTY}'),
    ('specialties_meeting_threshold', f'{N_SPECIALTIES_ELIGIBLE}'),
    ('prescribers_in_analysis_panel', f'{N_PRESCRIBERS_PANEL:,}'),
    ('panel_share_of_prescribers_pct', f'{100 * N_PRESCRIBERS_PANEL / N_PRESCRIBERS_ALL:.2f}'),
    ('panel_share_of_drug_cost_pct', f'{PANEL_COST_PCT:.2f}'),
    ('outlier_rule', f'Tukey IQR, {IQR_MULTIPLIER} x IQR fences, computed within specialty'),
    ('regression_unit', 'prescriber-drug row'),
    ('regression_n_obs', f'{N_OBS:,}'),
    ('regression_category_levels_total', f'{FULL_N_LEVELS:,}'),
    ('regression_n_free_parameters', f'{FULL_N_PARAMS:,}'),
    ('regression_estimator', 'OLS, unweighted, additive categorical main effects'),
], columns=['item', 'value'])

results_md = f"""# PY_RESULTS — Medicare Part D, Colorado 2024

Generated by `outputs/part_d_analysis.ipynb`. Numbers only.

## Table 1. Run facts

{to_md(run_facts)}

## Table 2. Prescriber-level metric distributions (analysis panel, n = {N_PRESCRIBERS_PANEL:,})

{to_md(metric_dist_out)}

## Table 3. Outlier flag counts

Rule: within each specialty, Q1 and Q3 of the metric across that specialty's
prescribers; IQR = Q3 - Q1; flag high if metric > Q3 + {IQR_MULTIPLIER} x IQR,
flag low if metric < Q1 - {IQR_MULTIPLIER} x IQR. Applied separately to each metric.

{to_md(flag_summary)}

## Table 4. Per-specialty rule parameters and high-side flag counts (all {N_SPECIALTIES_ELIGIBLE} specialties)

{to_md(spec_rule)}

## Table 5. Top 20 prescribers by cost-per-claim

{to_md(top_cpc)}

## Table 6. Top 20 prescribers by cost-per-30-day-fill

{to_md(top_cp30)}

## Table 7. Regression model fit

Outcome: cost-per-claim (Tot_Drug_Cst / Tot_Clms) at the prescriber-drug row level,
n = {N_OBS:,}. OLS, unweighted, additive categorical main effects.

{to_md(model_table)}

## Table 8. Variance explained by dimension

own_r_squared = R-squared of that dimension alone.
incremental_r_squared = full-model R-squared minus the R-squared of the model
containing the other two dimensions only.

{to_md(variance_table)}

Dimension with the largest single-dimension R-squared: **{WIN_OWN_DIM}**
(R-squared = {WIN_OWN_R2:.6f}).
Dimension with the largest incremental R-squared: **{WIN_INC_DIM}**
(incremental R-squared = {WIN_INC_R2:.6f}).
Full-model R-squared = {R2_FULL:.6f}. Unexplained share = {1 - R2_FULL:.6f}.
Sum of single-dimension R-squareds minus full-model R-squared = {shared:.6f}.

## Table 9. Regression coefficients — intercept and all specialty levels

Reference levels (omitted, folded into the intercept): specialty =
`{REF_SPECIALTY}`, city = `{REF_CITY}`, generic = `{REF_GENERIC}`.
Units: dollars per claim.

{to_md(coefs_specialty_out)}

## Table 10. Regression coefficients — 20 largest and 20 smallest city levels

{to_md(city_extremes_out)}

## Table 11. Regression coefficients — 25 largest and 25 smallest generic-drug levels

{to_md(generic_extremes_out)}

Full coefficient set ({len(coefs):,} rows): `outputs/regression_output.csv`.
Full prescriber table ({len(outliers):,} rows): `outputs/outliers.csv`.
"""

(OUT / 'PY_RESULTS.md').write_text(results_md, encoding='utf-8')
print(f'wrote {OUT / "PY_RESULTS.md"}  ({len(results_md):,} chars)')

wrote C:\Users\Caleb Edwards\OneDrive\Documents\DAB Capstone\outputs\PY_RESULTS.md  (31,337 chars)


In [17]:
# Counts cited in PY_FLAGS, computed rather than asserted.
_lvl = coefs.set_index(['block', 'level'])['n_obs_at_level']
SPARSE = {b: (int((coefs[coefs['block'] == b]['n_obs_at_level'] <= 2).sum()),
              int(coefs[coefs['block'] == b].shape[0]),
              int(coefs[(coefs['block'] == b) & (coefs['n_obs_at_level'] <= 2)]['n_obs_at_level'].sum()))
          for b in ['specialty', 'city', 'generic']}
print('levels with <= 2 rows (n_sparse, n_levels, rows_covered):', SPARSE)

# Candidate spelling variants among city levels: both spellings are present as
# separate regression levels. Only pairs actually found in the file are listed.
_candidate_city_variants = [
    ('Colorado Springs', 'Colo Springs'), ('Lone Tree', 'Lone Treet'),
    ('Greeley', 'Greely'), ('Haxtun', 'Haxton'),
    ('Wheat Ridge', 'Wheatridge'), ('Greenwood Village', 'Greenwood Vlg'),
]
city_variant_lines = []
for a, b in _candidate_city_variants:
    if ('city', a) in _lvl.index and ('city', b) in _lvl.index:
        city_variant_lines.append(
            f'  - `{a}` ({int(_lvl[("city", a)]):,} rows) and `{b}` ({int(_lvl[("city", b)]):,} rows)')
CITY_VARIANTS = '\n'.join(city_variant_lines)
print(CITY_VARIANTS)

n_zero_cost_prescribers = int((pres['Tot_Drug_Cst'] == 0).sum())
median_rows_per_prescriber = float(pres['n_drug_rows'].median())
type_src_mix = raw['Prscrbr_Type_Src'].value_counts()
type_src_lines = '\n'.join(
    f'  - `{k}`: {v:,} rows ({100 * v / N_ROWS_SOURCE:.1f}%)' for k, v in type_src_mix.items())
y_max = float(y.max())
y_p50 = float(np.median(y))

flags_md = f"""# PY_FLAGS — Medicare Part D, Colorado 2024

Companion to `PY_RESULTS.md`. Nothing here is needed to read the results tables.
Sections: data quality, foreseeable transformations (flagged, not built),
model-specification concerns, ambiguities and the choices made, and
interpretation that entered the work.

## 1. Data quality

- **CMS suppression sets a floor on what is visible.** Every row in the file has
  `Tot_Clms` >= 11, the CMS cell-suppression threshold, and the minimum
  `Tot_30day_Fills` in the file is likewise 11. Prescriber-drug combinations below
  that threshold are absent entirely. Prescriber totals computed here are
  therefore totals over reported rows, not the prescriber's true totals, and the
  gap is larger for prescribers whose volume is spread thinly across many drugs.
  Both primary metrics inherit this.
- **Beneficiary columns are unusable as denominators.** `Tot_Benes` is blank in
  58.9% of rows, `GE65_Tot_Benes` in 86.3%, and the GE65 claim/cost/fill columns
  in 42.1%. The blanks are suppression, not missingness at random. No metric here
  uses them; cost-per-beneficiary is not computed anywhere in this notebook.
- **{N_ZERO_COST_ROWS} rows report `Tot_Drug_Cst` = 0** with non-zero claims,
  giving cost-per-claim of exactly 0. No negative costs appear. These rows were
  retained unaltered, per the no-cleaning constraint. They sit at the outcome's
  lower bound in the regression and pull the low tail of the outlier screen.
  Prescribers whose total drug cost is 0 after aggregation: {n_zero_cost_prescribers}.
- **Specialty labels come from three different sources** (`Prscrbr_Type_Src`):
{type_src_lines}
  The label is not uniformly derived, so peer groups mix claim-derived and
  registry-derived specialty assignments. `Prscrbr_Type_Src` is carried into
  `outliers.csv` so the mix is visible per prescriber.
- **City is the free-text value as filed** (226 distinct values). No
  normalization, deduplication, or geocoding was applied, and the level set
  contains pairs that look like spellings of the same place, each carried as its
  own regression level:
{CITY_VARIANTS}
  Reading each pair as one place is an inference, not something the file states;
  it is repeated under Section 5. The low-count spelling in each pair also
  produces a coefficient fit to very few rows.
- **Many levels are supported by very few rows.** Counting levels with 2 or fewer
  rows: {SPARSE['generic'][0]:,} of {SPARSE['generic'][1]:,} generic drugs
  ({SPARSE['generic'][2]:,} rows), {SPARSE['city'][0]:,} of {SPARSE['city'][1]:,}
  cities ({SPARSE['city'][2]:,} rows), {SPARSE['specialty'][0]:,} of
  {SPARSE['specialty'][1]:,} specialties ({SPARSE['specialty'][2]:,} rows).
  Coefficients on those levels are fit to a handful of observations and are not
  stable quantities. `n_obs_at_level` is in `regression_output.csv` for every
  level so they can be screened out.
- **Generic name does not encode strength, dosage form, or route.** 1,177
  distinct `Gnrc_Name` values cover multiple strengths and formulations of the
  same molecule, which vary widely in cost per claim. The file has no NDC-level
  detail to separate them.
- **Each NPI maps to exactly one specialty and one city in this file**
  (asserted in the notebook), so the prescriber-level carry-forward of those
  labels is lossless.
- Aggregation to prescriber level preserves the cost total exactly:
  ${TOTAL_COST_SOURCE:,.2f} before, ${agg_total:,.2f} after.

## 2. Transformations foreseeable at capstone time (flagged, not built)

Per the no-transformation constraint, none of these were applied.

- **Log or Box-Cox transform of cost-per-claim.** The row-level outcome runs from
  0 to ${y_max:,.2f} with a median of ${y_p50:,.2f}. A level-scale OLS is fit here
  as specified; a log-scale fit is the conventional treatment and would change
  both the coefficients and the R-squared. The {N_ZERO_COST_ROWS} zero-cost rows
  would need an offset or an explicit exclusion rule first.
- **Claim-weighted estimation.** The regression is unweighted, so an 11-claim row
  and a 50,000-claim row count equally. Weighting by `Tot_Clms` is the natural
  alternative and would change every coefficient.
- **Winsorizing or trimming the extreme tail** of both metrics before fitting.
- **City-name normalization.** The apparent spelling variants listed in Section 1
  would be merged before any city-level modelling. Not done here.
- **Collapsing rare levels.** Many cities and generics appear in very few rows;
  pooling them into an `Other` category would cut the parameter count and the
  overfitting risk. Level counts are in `regression_output.csv`.
- **Brand-versus-generic indicator.** `Brnd_Name` and `Gnrc_Name` are both
  present and their relationship is not encoded as a variable anywhere here.
- **Days-supply normalization.** `Tot_Day_Suply` was not loaded or used; a
  cost-per-day-supply metric is available in the source and was not built.
- **Specialty label reconciliation** across the three `Prscrbr_Type_Src` values.

## 3. Model-specification concerns

- **Errors are not independent.** A prescriber contributes many rows (median
  {median_rows_per_prescriber:.0f}) and each drug appears across many
  prescribers. No clustered, robust, or panel-corrected standard errors were
  computed. This is the single largest specification problem with the model.
- **No standard errors, t-statistics, or p-values are reported at all.** The
  {FULL_N_PARAMS:,}-parameter design was fit by alternating projections, which yields point
  estimates without `(X'X)^-1`; a sparse linear-algebra library is not available
  in this environment. Every coefficient in `regression_output.csv` is a point
  estimate with no reported uncertainty, and none should be read as significant
  or not significant.
- **Heteroskedasticity is near-certain** given the outcome's skew, and is not
  addressed.
- **Main effects only.** The model assumes a specialty shifts cost-per-claim by
  the same dollar amount regardless of drug or city. Interactions are not fit.
- **R-squared rises mechanically with level count.** The generic-drug factor has
  1,177 levels against 226 cities and 97 specialties, so part of its advantage in
  Table 8 is a degrees-of-freedom artifact. Adjusted R-squared is reported next
  to it for that reason; with n = {N_OBS:,} the adjustment is small.
- **The three dimensions overlap**, so the single-dimension R-squareds do not
  partition the variance and their sum exceeds the full-model R-squared by
  {shared:.6f}. Incremental R-squared is reported alongside for this reason;
  neither measure alone is a clean attribution.
- **The outcome is bounded below at 0** and {N_ZERO_COST_ROWS} observations are
  at the bound, which a linear model does not respect.
- **No causal identification is attempted or possible here.** The coefficients
  are conditional mean differences within this file. Nothing is controlled for
  beyond the three categorical dimensions, and patient case-mix, which plainly
  drives drug choice, is not observed in this dataset at all.
- **Two different units of analysis appear in this notebook.** The outlier screen
  is prescriber-level ({N_PRESCRIBERS_PANEL:,} rows); the regression is
  prescriber-drug-row-level ({N_OBS:,} rows). Results from the two sections are
  not directly comparable.

## 4. Ambiguities and the choices made

- **"Cost-per-claim as outcome" is unit-ambiguous.** It could mean the
  prescriber-level ratio or the row-level ratio. Row level was chosen because
  generic drug is a row-level attribute and cannot enter a prescriber-level
  model. Consequence: the regression is fit on a different unit than the outlier
  screen.
- **"Variance-explained comparison" is not a single defined quantity** when
  predictors are correlated. Two measures are reported (single-dimension
  R-squared and incremental R-squared) rather than picking one. They agree on the
  ranking in this run.
- **The >= 30 prescribers restriction was applied at prescriber level** after
  aggregation, which yields 46 specialties as the brief states. Applying it to
  row counts would give a different set.
- **Prescriber metrics were built as summed-cost over summed-denominator**, not
  as the mean of that prescriber's row-level ratios. The two differ; the former
  is claim-weighted and preserves the cost total.
- **Both tails are flagged** by the outlier rule. The brief did not specify a
  direction, so high and low are flagged separately and reported separately.
- **Percentile ranks use average ranking for ties** (pandas default) and are
  expressed 0-100 within specialty.
- **Reference levels for the regression are the first level alphabetically** in
  each dimension, which is arbitrary. Coefficients are differences versus those
  levels and change meaning entirely if the reference changes; the reference
  levels are marked in `regression_output.csv`.
- **`part_d.sqlite` and `load_part_d.py` already in `outputs/` were not read or
  used.** This notebook reads the raw CSV only.

## 5. Interpretation that entered the work, labeled as such

The brief restricts output to calculations. These are the points where judgment,
not arithmetic, determined a result. They are listed here and kept out of
`PY_RESULTS.md`.

- **Choosing IQR fences over a z-score rule is a judgment**, made because both
  metric distributions are strongly right-skewed and heavy-tailed, so the mean
  and standard deviation a z-rule depends on are themselves inflated by the
  values being screened for. That reasoning is an interpretation of the observed
  distribution shape, not a result derived from it. The 1.5 multiplier is
  convention, not an estimate, and a different multiplier gives a different
  flagged set.
- **"Outlier" is a statistical label only.** A flag in `outliers.csv` means a
  prescriber's metric fell outside the fences for their specialty. It carries no
  claim about appropriateness, quality, or anything else, and the notebook makes
  no such claim.
- **Choosing the prescriber-drug row as the regression unit** is a modeling
  judgment (see Section 4), not something the data determined.
- **The three dimensions overlap for substantive reasons**, e.g. specialty and
  drug choice are related by clinical practice. That statement is interpretation;
  what the notebook actually computes is the numerical overlap of
  {shared:.6f} in Table 8.
- **Describing the outcome's distribution as "strongly skewed"** is a
  characterization; the underlying numbers (skew and kurtosis) are in
  `PY_RESULTS.md` Table 2.
- **Calling the city pairs in Section 1 spelling variants of one place is an
  inference.** What the file contains is distinct string values; nothing in it
  says they refer to the same municipality. No merge was performed on that basis.
"""

(OUT / 'PY_FLAGS.md').write_text(flags_md, encoding='utf-8')
print(f'wrote {OUT / "PY_FLAGS.md"}  ({len(flags_md):,} chars)')

levels with <= 2 rows (n_sparse, n_levels, rows_covered):

 {'specialty': (8, 97, 13), 'city': (24, 226, 30), 'generic': (251, 1177, 347)}
  - `Colorado Springs` (40,743 rows) and `Colo Springs` (9 rows)
  - `Lone Tree` (7,285 rows) and `Lone Treet` (65 rows)
  - `Greeley` (9,508 rows) and `Greely` (6 rows)
  - `Haxtun` (36 rows) and `Haxton` (64 rows)
  - `Wheat Ridge` (8,294 rows) and `Wheatridge` (63 rows)
  - `Greenwood Village` (4,176 rows) and `Greenwood Vlg` (8 rows)


wrote C:\Users\Caleb Edwards\OneDrive\Documents\DAB Capstone\outputs\PY_FLAGS.md  (10,898 chars)


In [18]:
# Final verification of the stated quality checks.
checks = [
    ('prescriber aggregation preserves total cost of $2.74B',
     abs(agg_total - 2_737_455_388.61) < 0.01, f'${agg_total:,.2f}'),
    ('specialty restriction yields 46 specialties, not 97',
     N_SPECIALTIES_ELIGIBLE == 46 and N_SPECIALTIES_ALL == 97,
     f'{N_SPECIALTIES_ELIGIBLE} of {N_SPECIALTIES_ALL}'),
    ('source row count unchanged (no cleaning/reshaping)',
     N_ROWS_SOURCE == 390_473 and N_OBS == 390_473, f'{N_OBS:,}'),
    ('regression converged (normal equations satisfied)',
     model_table['normal_eq_max_abs'].max() < 1e-3,
     f'{model_table["normal_eq_max_abs"].max():.2e}'),
    ('outliers.csv written', (OUT / 'outliers.csv').exists(),
     f'{(OUT / "outliers.csv").stat().st_size:,} bytes'),
    ('regression_output.csv written', (OUT / 'regression_output.csv').exists(),
     f'{(OUT / "regression_output.csv").stat().st_size:,} bytes'),
    ('PY_RESULTS.md written', (OUT / 'PY_RESULTS.md').exists(),
     f'{(OUT / "PY_RESULTS.md").stat().st_size:,} bytes'),
    ('PY_FLAGS.md written', (OUT / 'PY_FLAGS.md').exists(),
     f'{(OUT / "PY_FLAGS.md").stat().st_size:,} bytes'),
]
check_table = pd.DataFrame(checks, columns=['check', 'passed', 'value'])
display(check_table)
assert check_table['passed'].all(), check_table[~check_table['passed']]
print('all checks passed')

,check,passed,value
0,prescriber aggregation preserves total cost of...,True,"$2,737,455,388.61"
1,"specialty restriction yields 46 specialties, n...",True,46 of 97
2,source row count unchanged (no cleaning/reshap...,True,"390,473"
3,regression converged (normal equations satisfied),True,7.15e-07
4,outliers.csv written,True,"8,249,552 bytes"
5,regression_output.csv written,True,"101,250 bytes"
6,PY_RESULTS.md written,True,"31,738 bytes"
7,PY_FLAGS.md written,True,"11,079 bytes"


all checks passed
